In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 13.0 MB/s eta 0:00:00


In [2]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [6]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2026-07-10 10:43:06,779] A new study created in memory with name: no-name-02f35b7f-2001-4eb2-9282-e10f52b67697
[I 2026-07-10 10:43:10,655] Trial 0 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 95, 'max_depth': 20}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-07-10 10:43:18,376] Trial 1 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 175, 'max_depth': 8}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-07-10 10:43:27,788] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 174, 'max_depth': 7}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-07-10 10:43:30,248] Trial 3 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 52, 'max_depth': 14}. Best is trial 3 with value: 0.7746741154562384.
[I 2026-07-10 10:43:37,347] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 159, 'max_depth': 17}. Best is trial 3 with value: 0.77467411

In [7]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7783985102420856
Best hyperparameters: {'n_estimators': 128, 'max_depth': 13}


In [8]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


## Samplers in Optuna

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [10]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-07-10 10:44:28,839] A new study created in memory with name: no-name-dfcc39b0-0677-422c-b87d-c63eb46533c0
[I 2026-07-10 10:44:29,948] Trial 0 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 199, 'max_depth': 9}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-07-10 10:44:30,713] Trial 1 finished with value: 0.7541899441340782 and parameters: {'n_estimators': 158, 'max_depth': 3}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-07-10 10:44:31,099] Trial 2 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 66, 'max_depth': 17}. Best is trial 2 with value: 0.7690875232774674.
[I 2026-07-10 10:44:31,972] Trial 3 finished with value: 0.7541899441340782 and parameters: {'n_estimators': 179, 'max_depth': 3}. Best is trial 2 with value: 0.7690875232774674.
[I 2026-07-10 10:44:32,954] Trial 4 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 132, 'max_depth': 4}. Best is trial 2 with value: 0.769087523

In [11]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7783985102420857
Best hyperparameters: {'n_estimators': 133, 'max_depth': 12}


In [12]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


In [13]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [14]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-07-10 10:45:08,081] A new study created in memory with name: no-name-ed98d66c-f75c-477d-a845-ddedb849d6e0
[I 2026-07-10 10:45:08,608] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-07-10 10:45:09,452] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-07-10 10:45:09,741] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-07-10 10:45:10,326] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-07-10 10:45:10,915] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [15]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [16]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


## Optuna Visualizations

In [17]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [18]:
# 1. Optimization History
plot_optimization_history(study).show()

In [19]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [20]:
# 3. Slice Plot
plot_slice(study).show()

In [21]:
# 4. Contour Plot
plot_contour(study).show()

In [22]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

## Optimizing Multiple ML Models

In [23]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [24]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [25]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-07-10 10:45:25,109] A new study created in memory with name: no-name-480fd657-4f0b-4c24-b01b-116b094ede6b
[I 2026-07-10 10:45:25,179] Trial 0 finished with value: 0.7392923649906891 and parameters: {'classifier': 'SVM', 'C': 28.69385477782629, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0.7392923649906891.
[I 2026-07-10 10:45:27,018] Trial 1 finished with value: 0.7746741154562384 and parameters: {'classifier': 'RandomForest', 'n_estimators': 280, 'max_depth': 6, 'min_samples_split': 8, 'min_samples_leaf': 2, 'bootstrap': False}. Best is trial 1 with value: 0.7746741154562384.
[I 2026-07-10 10:45:30,973] Trial 2 finished with value: 0.7541899441340782 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 268, 'learning_rate': 0.01569166536170524, 'max_depth': 16, 'min_samples_split': 7, 'min_samples_leaf': 6}. Best is trial 1 with value: 0.7746741154562384.
[I 2026-07-10 10:45:31,019] Trial 3 finished with value: 0.7765363128491619 and parameters

In [26]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.11895011990420502, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [27]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.739292,2026-07-10 10:45:25.112094,2026-07-10 10:45:25.179239,0 days 00:00:00.067145,28.693855,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.774674,2026-07-10 10:45:25.181670,2026-07-10 10:45:27.018232,0 days 00:00:01.836562,NaN,False,RandomForest,NaN,NaN,NaN,6.0,2.0,8.0,280.0,COMPLETE
2,2,0.754190,2026-07-10 10:45:27.019266,2026-07-10 10:45:30.973226,0 days 00:00:03.953960,NaN,NaN,GradientBoosting,NaN,NaN,0.015692,16.0,6.0,7.0,268.0,COMPLETE
3,3,0.776536,2026-07-10 10:45:30.974355,2026-07-10 10:45:31.019722,0 days 00:00:00.045367,0.195580,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.787709,2026-07-10 10:45:31.022400,2026-07-10 10:45:31.048725,0 days 00:00:00.026325,0.178129,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.785847,2026-07-10 10:45:59.746356,2026-07-10 10:45:59.775130,0 days 00:00:00.028774,0.194546,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.765363,2026-07-10 10:45:59.776105,2026-07-10 10:46:00.552336,0 days 00:00:00.776231,NaN,NaN,GradientBoosting,NaN,NaN,0.136301,5.0,5.0,5.0,94.0,COMPLETE
97,97,0.787709,2026-07-10 10:46:00.553346,2026-07-10 10:46:00.581577,0 days 00:00:00.028231,0.101412,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.787709,2026-07-10 10:46:00.582525,2026-07-10 10:46:00.610768,0 days 00:00:00.028243,0.157729,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [28]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,80
RandomForest,10
GradientBoosting,10


In [29]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.744134
RandomForest,0.764991
SVM,0.779120


In [30]:
# 1. Optimization History
plot_optimization_history(study).show()

In [31]:
# 3. Slice Plot
plot_slice(study).show()

In [32]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [36]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


[I 2026-07-10 10:57:01,706] A new study created in memory with name: no-name-ae539e3c-6aed-4e7f-81ea-8098752bc47f


[0]	train-mlogloss:0.86442	eval-mlogloss:0.86665
[1]	train-mlogloss:0.66838	eval-mlogloss:0.65954
[2]	train-mlogloss:0.53653	eval-mlogloss:0.51544
[3]	train-mlogloss:0.44307	eval-mlogloss:0.42004
[4]	train-mlogloss:0.36790	eval-mlogloss:0.33833
[5]	train-mlogloss:0.30963	eval-mlogloss:0.27808
[6]	train-mlogloss:0.26899	eval-mlogloss:0.23691
[7]	train-mlogloss:0.24592	eval-mlogloss:0.21317
[8]	train-mlogloss:0.21492	eval-mlogloss:0.17685
[9]	train-mlogloss:0.18873	eval-mlogloss:0.14504
[10]	train-mlogloss:0.16678	eval-mlogloss:0.12011
[11]	train-mlogloss:0.14909	eval-mlogloss:0.10271
[12]	train-mlogloss:0.13958	eval-mlogloss:0.09128
[13]	train-mlogloss:0.13094	eval-mlogloss:0.08039
[14]	train-mlogloss:0.12362	eval-mlogloss:0.07138
[15]	train-mlogloss:0.11804	eval-mlogloss:0.06437
[16]	train-mlogloss:0.11302	eval-mlogloss:0.05780
[17]	train-mlogloss:0.10848	eval-mlogloss:0.05386
[18]	train-mlogloss:0.10367	eval-mlogloss:0.04945
[19]	train-mlogloss:0.10156	eval-mlogloss:0.04725
[20]	train

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning:

`optuna.integration.xgboost` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.xgboost` instead.



[51]	train-mlogloss:0.08532	eval-mlogloss:0.03681
[52]	train-mlogloss:0.08539	eval-mlogloss:0.03697
[53]	train-mlogloss:0.08537	eval-mlogloss:0.03696
[54]	train-mlogloss:0.08532	eval-mlogloss:0.03685
[55]	train-mlogloss:0.08530	eval-mlogloss:0.03688
[56]	train-mlogloss:0.08530	eval-mlogloss:0.03688
[57]	train-mlogloss:0.08531	eval-mlogloss:0.03693
[58]	train-mlogloss:0.08530	eval-mlogloss:0.03693
[59]	train-mlogloss:0.08527	eval-mlogloss:0.03691
[60]	train-mlogloss:0.08527	eval-mlogloss:0.03688
[61]	train-mlogloss:0.08391	eval-mlogloss:0.03777
[62]	train-mlogloss:0.08391	eval-mlogloss:0.03778
[63]	train-mlogloss:0.08391	eval-mlogloss:0.03780
[64]	train-mlogloss:0.08390	eval-mlogloss:0.03787
[65]	train-mlogloss:0.08390	eval-mlogloss:0.03793
[66]	train-mlogloss:0.08390	eval-mlogloss:0.03786
[67]	train-mlogloss:0.08274	eval-mlogloss:0.03561
[68]	train-mlogloss:0.08274	eval-mlogloss:0.03561
[69]	train-mlogloss:0.08273	eval-mlogloss:0.03555
[70]	train-mlogloss:0.08274	eval-mlogloss:0.03556


[I 2026-07-10 10:57:02,173] Trial 0 finished with value: 1.0 and parameters: {'lambda': 0.012387904356519869, 'alpha': 0.08624542710734392, 'eta': 0.21995919573520484, 'gamma': 0.9461755926977051, 'max_depth': 9, 'min_child_weight': 1, 'subsample': 0.8296331284679861, 'colsample_bytree': 0.6944005548856986}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:1.07050	eval-mlogloss:1.07314
[1]	train-mlogloss:1.04269	eval-mlogloss:1.04546
[2]	train-mlogloss:1.02716	eval-mlogloss:1.03516
[3]	train-mlogloss:1.01265	eval-mlogloss:1.02378


[I 2026-07-10 10:57:07,508] Trial 48 pruned. Trial was pruned at iteration 4.


[0]	train-mlogloss:0.81935	eval-mlogloss:0.80064


[I 2026-07-10 10:57:07,543] Trial 49 pruned. Trial was pruned at iteration 1.


Best trial: {'lambda': 0.012387904356519869, 'alpha': 0.08624542710734392, 'eta': 0.21995919573520484, 'gamma': 0.9461755926977051, 'max_depth': 9, 'min_child_weight': 1, 'subsample': 0.8296331284679861, 'colsample_bytree': 0.6944005548856986}
Best accuracy: 1.0


In [35]:
! pip install optuna-integration[xgboost]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 1.1 MB/s eta 0:00:00


In [37]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()